# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khuld13/ML-intern-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

In [2]:
%cd /content
!rm -rf ML-intern-starter
!git clone https://github.com/Khuld13/ML-intern-starter.git
%cd ML-intern-starter

/content
Cloning into 'ML-intern-starter'...
remote: Enumerating objects: 201, done.
remote: Counting objects: 100% (201/201), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 201 (delta 99), reused 88 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (201/201), 1.92 MiB | 11.34 MiB/s, done.
Resolving deltas: 100% (99/99), done.
/content/ML-intern-starter


In [3]:
!cat scripts/03_*.py 2>/dev/null | grep -A2 "read_parquet\|hf://"

In [4]:
!cat <path_from_above> | grep -B2 -A5 "read_parquet\|hf://"

/bin/bash: -c: line 1: syntax error near unexpected token `|'
/bin/bash: -c: line 1: `cat <path_from_above> | grep -B2 -A5 "read_parquet\|hf://"'


In [5]:
!grep -n "read_parquet\|hf://" notebooks/03_working_with_the_full_release.ipynb

55:    "DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the\n",
70:    "REL = 'hf://datasets/FlyRank/internship-warehouse'\n",
72:    "    'dim_clients':                f\"read_parquet('{REL}/dim_clients.parquet')\",\n",
73:    "    'dim_content':                f\"read_parquet('{REL}/dim_content.parquet')\",\n",
74:    "    'fact_daily':                 f\"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')\",\n",
75:    "    'fact_daily_sample':          f\"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')\",\n",
76:    "    'fact_query_90d':             f\"read_parquet('{REL}/fact_content_query_90d.parquet')\",\n",


In [6]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

hf_token = userdata.get('HF_TOKEN')
con.sql(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
);
""")

REL = 'hf://datasets/FlyRank/internship-warehouse'

con.sql(f"""
CREATE OR REPLACE VIEW dim_content AS
SELECT * FROM read_parquet('{REL}/dim_content.parquet');
""")

con.sql(f"""
CREATE OR REPLACE VIEW fact_daily AS
SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03';
""")

print("Connected. Views ready.")

Connected. Views ready.


In [7]:
con.sql("SELECT COUNT(*) FROM dim_content").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       519606 │
└──────────────┘



In [8]:
con.sql("""
    CREATE OR REPLACE VIEW fact_march AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""")

con.sql("""
    CREATE OR REPLACE VIEW fact_feb AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet')
""")

print("fact_march and fact_feb ready.")

fact_march and fact_feb ready.


In [9]:
# Reconstruct ML-07's exact population and label (cell 8 logic from w04_baseline_score.ipynb)

monthly_compare = con.sql("""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM fact_march
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    ),
    feb_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM fact_feb
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    )
    SELECT
        m.content_hash_id,
        f.impressions_feb,
        m.impressions_march,
        CASE WHEN m.impressions_march < f.impressions_feb THEN 1 ELSE 0 END AS declined_flag
    FROM march_agg m
    JOIN feb_agg f USING (content_hash_id)
""").df()

# ML-07's volume-floor rule (Signal 2, CONFIRMED)
pop = monthly_compare[monthly_compare['impressions_march'] >= 250].copy()

# Bring in content-level features to model with
dim = con.sql("SELECT * FROM dim_content").df()
df = pop.merge(dim, on='content_hash_id', how='left')

# Exclude: the label itself, the two raw inputs that DEFINE the label,
# and the known-leaky/known-invalid columns from ML-06
leakage_cols = [
    'declined_flag', 'impressions_feb', 'impressions_march',   # label + its direct inputs
    'trend_pct', 'trend_direction', 'is_declining_label',       # ML-06: same fact, 3 forms
    'days_since_update',                                        # ML-06: structurally invalid (July snapshot)
]
candidate_features = [c for c in df.columns if c not in leakage_cols]

print("Rows after volume filter (impressions_march >= 250):", len(df))
print("\nLabel balance (declined_flag):")
print(df['declined_flag'].value_counts(normalize=True))
print("\nCandidate feature count:", len(candidate_features))
print(candidate_features)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after volume filter (impressions_march >= 250): 68581

Label balance (declined_flag):
declined_flag
0    0.771759
1    0.228241
Name: proportion, dtype: float64

Candidate feature count: 26
['content_hash_id', 'client_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [10]:
print(df[['optimization_eligible_date', 'last_optimized_date', 'provider_used', 'model_used']].describe(include='all'))
print(df[['optimization_eligible_date', 'last_optimized_date']].head(10))

        optimization_eligible_date         last_optimized_date provider_used  \
count                        31980                       31980          9310   
unique                         NaN                         NaN             4   
top                            NaN                         NaN        google   
freq                           NaN                         NaN          8339   
mean    2026-07-24 01:12:45.928705  2026-06-09 01:12:45.928705           NaN   
min            2026-06-08 00:00:00         2026-04-24 00:00:00           NaN   
25%            2026-07-10 00:00:00         2026-05-26 00:00:00           NaN   
50%            2026-07-26 00:00:00         2026-06-11 00:00:00           NaN   
75%            2026-08-06 00:00:00         2026-06-22 00:00:00           NaN   
max            2026-08-20 00:00:00         2026-07-06 00:00:00           NaN   

                    model_used  
count                    52282  
unique                       5  
top     gemini-3-fla

## 1. Method choice and why

**Method: Logistic regression, with `class_weight='balanced'`.**

**Population:** Reconstructed ML-07's exact rule population — content items present in
both `fact_feb` and `fact_march` with `gsc_data_available = TRUE`, filtered to
`impressions_march >= 250` (ML-07's Signal 2, confirmed volume floor). This gives
68,581 rows.

**Label:** `declined_flag` (impressions_march < impressions_feb), identical to ML-07 —
not a future-window label. This model tests "can a learned method rank better than
ML-07's fixed rule on the same question," not "can we forecast decline."

**Class balance:** 22.8% positive (declined), 77.2% negative. Not severe, but real
enough that `class_weight='balanced'` is necessary — an unweighted model would default
toward predicting "no decline" and look falsely accurate.

**Leakage exclusions, from this population's own audit:**
- `impressions_feb` / `impressions_march` — these directly define the label, not just
  correlate with it.
- `optimization_eligible_date` — a forward-scheduling product field; its own max value
  (2026-08-20) sits weeks past the snapshot's export date (2026-07-03), confirming it's
  a rule-output, not an observed signal.
- `last_optimized_date` — mostly dated after the March decision point (min 2026-04-24),
  so using it would leak post-decision information backward into the features.
- `days_since_update` — carried over from ML-06: structurally invalid against most rows
  because `dim_content` is a single July snapshot.
- `trend_pct`, `trend_direction`, `is_declining_label` — ML-06 flagged these as leakage
  in the starter dataset, but they don't exist as columns in this warehouse table, so
  the exclusion is moot here rather than actively enforced.

26 candidate columns after removing join keys and the above, mostly numeric content
metadata (search_volume, competition, cpc, word_count, char_count) plus a few sparse
categoricals (`provider_used`, `model_used` — missing for 76-86% of rows, kept but
expected to contribute little).

**Why logistic regression fits:** 68,581 rows against ~20 usable features is well
within simple-linear-method territory — there's no evidence of the kind of nonlinear
interaction that would justify a tree ensemble's added complexity, and the rubric
explicitly rewards explainability over complexity for its own sake. Logistic
regression's coefficients also let me directly compare which signals move the needle
against ML-07's hand-picked rule (decline + volume only), which is the actual point of
this comparison. Decision tree is the fallback if logistic underperforms badly enough
to suggest real nonlinearity.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Unique clients:", df['client_hash_id'].nunique())
print("\nRows per client (distribution):")
print(df.groupby('client_hash_id').size().describe())
print("\nTop 10 clients by row count:")
print(df.groupby('client_hash_id').size().sort_values(ascending=False).head(10))

Unique clients: 34

Rows per client (distribution):
count       34.000000
mean      2017.088235
std       3867.955002
min          1.000000
25%         16.250000
50%        443.500000
75%       1234.750000
max      15862.000000
dtype: float64

Top 10 clients by row count:
client_hash_id
client_73cda7b4e4f265ea    15862
client_62f4a7e64f5e0096    12642
client_23a62021009f63c4     9830
client_e547b89c05043229     7482
client_fef1a8f436438636     5922
client_08a6a72ff48e62c0     3909
client_20259bd6705d81d4     2736
client_e5c2aa26a8598242     1972
client_3f0ce4d44fe94f3d     1249
client_a80fca3f171ed1de     1192
dtype: int64


In [12]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print("Train rows:", len(train_df), " Test rows:", len(test_df))
print("Train clients:", train_df['client_hash_id'].nunique(), " Test clients:", test_df['client_hash_id'].nunique())
print("\nTrain label balance:\n", train_df['declined_flag'].value_counts(normalize=True))
print("\nTest label balance:\n", test_df['declined_flag'].value_counts(normalize=True))

Train rows: 54873  Test rows: 13708
Train clients: 27  Test clients: 7

Train label balance:
 declined_flag
0    0.787036
1    0.212964
Name: proportion, dtype: float64

Test label balance:
 declined_flag
0    0.710607
1    0.289393
Name: proportion, dtype: float64


In [13]:
for seed in [0, 1, 7, 42, 99]:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_idx, te_idx = next(gss.split(df, groups=df['client_hash_id']))
    te = df.iloc[te_idx]
    print(f"seed={seed}: test_rows={len(te)}, test_clients={te['client_hash_id'].nunique()}, "
          f"test_decline_rate={te['declined_flag'].mean():.3f}")

seed=0: test_rows=17470, test_clients=7, test_decline_rate=0.300
seed=1: test_rows=7383, test_clients=7, test_decline_rate=0.144
seed=7: test_rows=15071, test_clients=7, test_decline_rate=0.227
seed=42: test_rows=13708, test_clients=7, test_decline_rate=0.289
seed=99: test_rows=3639, test_clients=7, test_decline_rate=0.038


In [14]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

fold_decline_rates = []
for fold, (tr_idx, te_idx) in enumerate(sgkf.split(df, y=df['declined_flag'], groups=df['client_hash_id'])):
    te = df.iloc[te_idx]
    fold_decline_rates.append(te['declined_flag'].mean())
    print(f"fold={fold}: test_rows={len(te)}, test_clients={te['client_hash_id'].nunique()}, "
          f"test_decline_rate={te['declined_flag'].mean():.3f}")

print(f"\nMean test decline rate across folds: {np.mean(fold_decline_rates):.3f}")
print(f"Std across folds: {np.std(fold_decline_rates):.3f}")

fold=0: test_rows=53, test_clients=1, test_decline_rate=0.321
fold=1: test_rows=11593, test_clients=6, test_decline_rate=0.203
fold=2: test_rows=9074, test_clients=12, test_decline_rate=0.140
fold=3: test_rows=29719, test_clients=10, test_decline_rate=0.236
fold=4: test_rows=18142, test_clients=5, test_decline_rate=0.275

Mean test decline rate across folds: 0.235
Std across folds: 0.062


In [15]:
sgkf3 = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
for fold, (tr_idx, te_idx) in enumerate(sgkf3.split(df, y=df['declined_flag'], groups=df['client_hash_id'])):
    te = df.iloc[te_idx]
    print(f"fold={fold}: test_rows={len(te)}, test_clients={te['client_hash_id'].nunique()}, "
          f"test_decline_rate={te['declined_flag'].mean():.3f}")

fold=0: test_rows=9475, test_clients=10, test_decline_rate=0.104
fold=1: test_rows=45705, test_clients=11, test_decline_rate=0.265
fold=2: test_rows=13401, test_clients=13, test_decline_rate=0.192


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

numeric_features = ['keyword_char_count', 'keyword_token_count', 'url_char_count',
                     'search_volume', 'competition', 'cpc', 'backlinks', 'category_count',
                     'char_count', 'word_count']
categorical_features = ['content_type', 'competition_level', 'main_intent',
                         'provider_used', 'model_used', 'is_published', 'is_deleted']

X = df[numeric_features + categorical_features].copy()
nullable_int_cols = ['search_volume', 'backlinks', 'char_count', 'word_count']
X[nullable_int_cols] = X[nullable_int_cols].astype('float64')

y = df['declined_flag']
groups = df['client_hash_id']
pct_decline = (df['impressions_feb'] - df['impressions_march']) / df['impressions_feb'].replace(0, np.nan)

print("X shape:", X.shape)
print("NaN counts:\n", X.isna().sum()[X.isna().sum() > 0])

X shape: (68581, 17)
NaN counts:
 search_volume          939
competition            939
cpc                    939
backlinks            28174
char_count           18640
word_count           18640
competition_level     1167
main_intent            773
provider_used        59271
model_used           16299
dtype: int64


In [19]:
from sklearn.impute import SimpleImputer

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

In [20]:
print(preprocessor.transformers[0])

('num', Pipeline(steps=[('impute', SimpleImputer(strategy='median')),
                ('scale', StandardScaler())]), ['keyword_char_count', 'keyword_token_count', 'url_char_count', 'search_volume', 'competition', 'cpc', 'backlinks', 'category_count', 'char_count', 'word_count'])


In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return y_true.iloc[order].mean()

sgkf3 = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

results = []
for fold, (tr_idx, te_idx) in enumerate(sgkf3.split(df, y=y, groups=groups)):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

    model.fit(X_tr, y_tr)
    model_probs = model.predict_proba(X_te)[:, 1]

    baseline_scores = np.where(y_te.values == 1, pct_decline.iloc[te_idx].fillna(0).values, 0)

    results.append({
        'fold': fold,
        'test_rows': len(te_idx),
        'model_p@50': precision_at_k(y_te, model_probs, 50),
        'baseline_p@50': precision_at_k(y_te, baseline_scores, 50),
        'model_auc': roc_auc_score(y_te, model_probs),
        'model_ap': average_precision_score(y_te, model_probs),
    })
    print(results[-1])

results_df = pd.DataFrame(results)
print("\n", results_df)
print("\nMean model P@50:", results_df['model_p@50'].mean().round(3))
print("Mean baseline P@50:", results_df['baseline_p@50'].mean().round(3))
print("Mean model AUC:", results_df['model_auc'].mean().round(3))
print("Mean model AP:", results_df['model_ap'].mean().round(3))

{'fold': 0, 'test_rows': 9475, 'model_p@50': np.float64(0.56), 'baseline_p@50': np.float64(1.0), 'model_auc': np.float64(0.5583249658376997), 'model_ap': np.float64(0.149030988035586)}
{'fold': 1, 'test_rows': 45705, 'model_p@50': np.float64(0.18), 'baseline_p@50': np.float64(1.0), 'model_auc': np.float64(0.5645916446539295), 'model_ap': np.float64(0.2938173291291976)}
{'fold': 2, 'test_rows': 13401, 'model_p@50': np.float64(0.16), 'baseline_p@50': np.float64(1.0), 'model_auc': np.float64(0.47650614267642455), 'model_ap': np.float64(0.16810154355160686)}

    fold  test_rows  model_p@50  baseline_p@50  model_auc  model_ap
0     0       9475        0.56            1.0   0.558325  0.149031
1     1      45705        0.18            1.0   0.564592  0.293817
2     2      13401        0.16            1.0   0.476506  0.168102

Mean model P@50: 0.3
Mean baseline P@50: 1.0
Mean model AUC: 0.533
Mean model AP: 0.204


In [22]:
# 1. Did logistic regression actually converge?
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    for warning in w:
        print(warning.category.__name__, ":", warning.message)

# 2. Sanity check: does the model produce ANY spread in its predictions,
# or is it just predicting near-constant probabilities for everyone?
print(pd.Series(model_probs).describe())

count    13401.000000
mean         0.482220
std          0.107733
min          0.002655
25%          0.443924
50%          0.492390
75%          0.528441
max          0.786722
dtype: float64


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The model shows weak, inconsistent signal across folds — mean AUC 0.533, ranging from
0.477 (fold 2, worse than chance) to 0.565 (fold 1). Inspecting fold 2 closely
(worst-performing) shows *why*: the largest coefficients by magnitude are almost
entirely sparse categorical dummies — `model_used_gpt-5-mini` (-1.18),
`content_type_feedly article` (1.14), `provider_used_None` (0.77, i.e. "creator
unknown," missing for 87% of rows) — rather than the more plausible, generalizable
numeric signals like `word_count` or `search_volume`, which rank lower. This is a
classic sparse-category overfitting pattern: a linear model with no regularization
tuning assigns large weight to rare categories based on a handful of examples, and
that weight doesn't generalize to the next client group.

The confusion matrix confirms this isn't just "weak" — it's mildly miscalibrated in
the wrong direction on this fold. Declined-class precision is 0.17, below the fold's
own base rate of 0.192 (2,568 of 13,401 rows) — meaning when the model predicts
"declined," it's right *less often than random guessing at the base rate* would be.

One reassuring detail: only 36 of 2,568 true decliners were confidently missed
(scored below 0.3 probability) — most sit in an uncertain middle near 0.5, not
systematically misclassified. That's consistent with "the feature set lacks real
separating signal" rather than "the model is wrong about a specific type of page,"
and rules out an obvious, fixable bug.

Precision@50 cannot be compared directly against the baseline's 1.000, because ML-07's
`meets_rule` collapses to `declined_flag==1` once the population is already filtered
to `impressions_march >= 250` — the baseline's positive predictions are a strict
subset of the ground truth by construction, making its Precision@50 tautological
rather than earned. AUC and Average Precision are the honest comparison metrics here.

**What this suggests:** `dim_content` describes *what a page is* (word count, CPC,
competition, content type, creator) rather than *how it has been performing*. No
prior-period impressions, clicks, position, or CTR trend made it into the feature set
— correctly excluded, since those are adjacent to or define the label itself. The
honest conclusion is that predicting month-over-month decline likely requires
behavioral/performance-history features from before the label window, which this
snapshot's static metadata doesn't provide. That's a finding about the data's limits,
not a fixable modeling mistake — and a more regularized model or a tree-based method
would likely still struggle with the same missing-information problem, since the
issue is what's available to learn from, not how it's learned.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Error analysis: what does the model lean on, and where is it wrong?

from sklearn.metrics import confusion_matrix, classification_report

# Fit on the full training population from fold 2 (the worst-performing fold) to inspect it closely
train_idx, test_idx = list(sgkf3.split(df, y=y, groups=groups))[2]
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

model.fit(X_tr, y_tr)
probs = model.predict_proba(X_te)[:, 1]
preds = (probs >= 0.5).astype(int)

# 1. What does the model lean on? Pull coefficients back out with their real feature names
feature_names = (
    numeric_features +
    list(model.named_steps['prep'].named_transformers_['cat']
         .named_steps['onehot'].get_feature_names_out(categorical_features))
)
coefs = model.named_steps['clf'].coef_[0]
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
print("Top 10 features by absolute coefficient (fold 2 fit):")
print(coef_df.sort_values('abs_coef', ascending=False).head(10)[['feature', 'coefficient']])

# 2. Where is it wrong? Confusion matrix at default 0.5 threshold
print("\nConfusion matrix (fold 2, threshold=0.5):")
print(confusion_matrix(y_te, preds))
print("\nClassification report:")
print(classification_report(y_te, preds, target_names=['not declined', 'declined']))

# 3. Does the model's ranking disagree with the baseline rule?
# Baseline flags EVERY declined page (by construction); check where the model
# ranks a true decliner LOW despite the baseline "catching" it perfectly
te_df = df.iloc[test_idx].copy()
te_df['model_prob'] = probs
declined_mask = y_te.values == 1
missed_by_model = te_df[declined_mask & (te_df['model_prob'] < 0.3)]
print(f"\nTrue decliners the model scored low (<0.3 probability): {len(missed_by_model)} of {declined_mask.sum()}")
print(missed_by_model[['content_hash_id', 'word_count', 'search_volume', 'competition', 'model_prob']].head(10))

Top 10 features by absolute coefficient (fold 2 fit):
                            feature  coefficient
28            model_used_gpt-5-mini    -1.178438
11      content_type_feedly article     1.139099
10  content_type_comparison article    -1.067944
22             provider_used_google    -0.841874
24               provider_used_None     0.767897
8                        char_count     0.718286
9                        word_count    -0.560221
30                  model_used_None     0.470950
25      model_used_gemini-2.5-flash     0.454943
19         main_intent_navigational    -0.433035

Confusion matrix (fold 2, threshold=0.5):
[[5885 4948]
 [1529 1039]]

Classification report:
              precision    recall  f1-score   support

not declined       0.79      0.54      0.65     10833
    declined       0.17      0.40      0.24      2568

    accuracy                           0.52     13401
   macro avg       0.48      0.47      0.44     13401
weighted avg       0.67      0.52      0.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.